# Background

Author: Diane Menuz  
Date: 2026-02-05  
Site: Dugout

Goal: Review qc data. Any data cleaning steps should be added back into the qc_data file. This is  
for data exploration only, not final data cleaning!!

# Setup

## Parameters

In [ ]:
station = "US-UTD"
station_name = 'dugout' # for outputing plots; create a subfolder with the name {stationname}_plots
date_range = '202305241430_202510221630'
micromet_path = r"C:/Users/dmenuz/Documents/scripts/MicroMet/src/"

parquet_path = r'M:\Shared drives\UGS_Flux\Data_Processing\final_database_tables' 
file_name = f'{station}_{date_range}_qc.parquet'
raw_file_name = f'{station}_{date_range}_raw.parquet'
micromet_path = r"C:/Users/dmenuz/Documents/scripts/MicroMet/src/"

## Libraries

In [ ]:
import sys
sys.path.append(micromet_path)
import micromet
from micromet import validate
from micromet import recalculate_albedo
from micromet import data_cleaning
from micromet import eddy_plots as ed_plot
from micromet import variable_limits as vl


In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import importlib

from prettytable import PrettyTable
from typing import Union

from scipy import stats
import matplotlib.pyplot as plt


from typing import List, Dict, Union

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from plotly.offline import iplot
from pathlib import Path

## Function to Test

In [ ]:
import plotly.graph_objs as go
from plotly.offline import iplot
import random

def plotlystuff_test(datasets, colnames, chrttypes=None, datatitles=None, chrttitle='', colors=None,
                two_yaxes=False, axislabels=None, opac=None, 
                plot_height=300):
    """
    Plots one or more datasets on a shared set of x-axes with support for dual y-axes.

    Parameters:
    -----------
    datasets : list of pd.DataFrame
        List of dataframes to plot. Must have a DatetimeIndex.
    colnames : list of str
        The column names to extract from each dataset.
    chrttypes : list of str, optional
        Plotly mode (e.g., 'lines', 'markers'). Defaults to 'lines'.
    datatitles : list of str, optional
        Legend names. Defaults to 'colnames'.
    chrttitle : str, optional
        Main chart title.
    colors : list of str, optional
        Hex codes for lines.
    two_yaxes : bool, optional
        If True, routes first dataset to left axis and subsequent to right.
    axislabels : list of str, optional
        Labels for [primary, secondary] y-axes. Defaults to first two 'colnames'.
    opac : list of float, optional
        Opacity (0 to 1). Defaults to 0.8.
    plot_height : int, optional
        Plot height in pixels. Defaults to 350.
    """
    
    n = len(datasets)
    chrttypes = chrttypes or ['lines'] * n
    opac = opac or [0.8] * n
    datatitles = datatitles or colnames
    
    # DEFAULT AXIS LABELS: Use first two colnames if not provided
    if axislabels is None:
        if two_yaxes and n >= 2:
            axislabels = [colnames[0], colnames[1]]
        else:
            axislabels = [colnames[0], ""]

    # Color Handling
    default_colors = ['#228B22', '#FF1493', '#5acafa', '#663399', '#FF0000']
    if colors is None:
        colors = (default_colors * (n // 5 + 1))[:n]

    # Axis Routing
    axis_mapping = ['y'] + (['y2'] * (n - 1)) if two_yaxes else ['y'] * n
    
    data_traces = []
    for i in range(n):
        trace = go.Scatter(
            x=datasets[i].index,
            y=datasets[i][colnames[i]],
            name=datatitles[i],
            line=dict(color=colors[i]),
            mode=chrttypes[i],
            opacity=opac[i],
            yaxis=axis_mapping[i]
        )
        data_traces.append(trace)
    
    layout = go.Layout(
        title=chrttitle,
        height=plot_height,
        xaxis=dict(rangeslider=dict(visible=True), type='date'),
        yaxis=dict(
            title=axislabels[0],
            titlefont=dict(color=colors[0]),
            tickfont=dict(color=colors[0])
        ),
        legend=dict(orientation="h", y=1.1, x=1, xanchor='right'),
        margin=dict(t=80, b=20, l=60, r=60),
        template='plotly_white'
    )
    
    if two_yaxes:
        layout['yaxis2'] = dict(
            title=axislabels[1],
            titlefont=dict(color=colors[1] if n > 1 else '#000000'),
            tickfont=dict(color=colors[1] if n > 1 else '#000000'),
            overlaying='y',
            side='right'
        )

    iplot({'data': data_traces, 'layout': layout})

# Read Data Back In

In [ ]:
file_path = Path (parquet_path, 'qc', file_name)
clean_temp = pd.read_parquet(file_path)
clean = clean_temp.copy()

# probably won't usually use raw data, but it is available if useful
file_path = Path (parquet_path, 'raw', raw_file_name)
raw_temp = pd.read_parquet(file_path)
raw = raw_temp.copy()

# NETRAD and SW/LW Comparison

## Daily Data

In [ ]:
daily_dat = clean[['NETRAD_1_1_1', 'NETRAD_1_1_2', 'days_since_20240101']].resample('D').mean()
color_col = 'days_since_20240101'


var1 = 'NETRAD_1_1_1'
var2 = 'NETRAD_1_1_2'
ed_plot.plot_interactive_regression_with_color(    
        daily_dat,
        var1, var2,
        color_col, plot_size=700)

In [ ]:
ed_plot.student_resid_plot(daily_dat, 'NETRAD_1_1_1', 'NETRAD_1_1_2', 'Daily NETRAD')

## Viewing and Data Cleanup

In [ ]:
ed_plot.plotlystuff([clean, clean], ['LW_IN_1_1_1', 'LW_IN_1_1_2'], 
            chrttitle='LW IN')
ed_plot.plotlystuff([clean, clean], ['LW_OUT_1_1_1', 'LW_OUT_1_1_2'], 
            chrttitle='LW Out')
ed_plot.plotlystuff([clean, clean], ['SW_IN_1_1_1', 'SW_IN_1_1_2'], 
            chrttitle='SW IN')
ed_plot.plotlystuff([clean, clean], ['SW_OUT_1_1_1', 'SW_OUT_1_1_2'], 
            chrttitle='SW OUT')
ed_plot.plotlystuff([clean, clean], ['NETRAD_1_1_1', 'NETRAD_1_1_2'], 
            chrttitle='NETRAD')

In [ ]:
# review linear regression plots 
vars = ['NETRAD',"LW_IN", 'LW_OUT', "SW_IN", 'SW_OUT']

for var in vars:
    var1 = f'{var}_1_1_1'
    var2 = f'{var}_1_1_2'
    color_col = 'days_since_20240101'

    ed_plot.plot_interactive_regression_with_color(    
        clean,
        var1, var2,
        color_col, plot_size=700)

In [ ]:
## PPFD_IN vs. SW_IN (from Ameriflux)
color_col = 'days_since_20240101'

ed_plot.plotlystuff([clean, clean, clean], 
                    ['SW_IN_1_1_1', 'SW_IN_1_1_1', 'PPFD_IN_1_1_1'], 
                    chrttitle='LW IN')

ed_plot.plot_interactive_regression_with_color(    
        clean,
        'SW_IN_1_1_1', 'PPFD_IN_1_1_1',
        color_col, plot_size=700)

ed_plot.plot_interactive_regression_with_color(    
        clean,
        'SW_IN_1_1_2', 'PPFD_IN_1_1_1',
        color_col, plot_size=700)

## NA in components with NETRAD values

In [ ]:
# does netrad have data when other variables are na
mask = ((clean.LW_IN_1_1_1.isna()) | (
    clean.LW_OUT_1_1_1.isna()) | (
        clean.SW_IN_1_1_1.isna()) | (
            clean.SW_OUT_1_1_1.isna())) & (~clean.NETRAD_1_1_1.isna())
subset1 = clean.loc[mask, [
    'LW_IN_1_1_1', 'LW_OUT_1_1_1', 'SW_IN_1_1_1', 'SW_OUT_1_1_1', 'NETRAD_1_1_1']]
print(f'NETRAD 1: {len(subset1)} records have NETRAD but missing one or more component values, equal to {(len(subset1)/len(mask)*100)} of records')

mask = ((clean.LW_IN_1_1_2.isna()) | (
    clean.LW_OUT_1_1_2.isna()) | (
        clean.SW_IN_1_1_2.isna()) | (
            clean.SW_OUT_1_1_2.isna())) & (
                ~clean.NETRAD_1_1_2.isna())
subset2 = clean.loc[mask, [
    'LW_IN_1_1_2', 'LW_OUT_1_1_2', 'SW_IN_1_1_2', 'SW_OUT_1_1_2', 'NETRAD_1_1_2']]
print(f'NETRAD 2: {len(subset2)} records have NETRAD but missing one or more component values, equal to {(len(subset2)/len(mask)*100)} of records')

## Albedo

- Calculated as SW_OUT/SW_IN
- In program, logic is as follows: 
   
If (R_SW_in > 0.1) AND (R_SW_in >= R_SW_out)  
    Then  
	albedo = 100.0*(R_SW_out/R_SW_in)   
	Else  
	albedo = 0.0  

**Notes**
- We changed the program to accept values as low as -20 for SW_IN and SW_OUT.
- Albedo values differ a lot when SW_IN or SW_OUT is very low, which makes sense bc values are so  
low. Not something we need to clean or address

In [ ]:
var1 = 'ALB_1_1_1'
var2 = 'ALB_1_1_2'
ed_plot.plot_interactive_regression_with_color(    
        clean,
        var1, var2,
        'SW_IN_1_1_1', plot_size=700)

ed_plot.plot_interactive_regression_with_color(    
        clean,
        var1, var2,
        'day_of_year', plot_size=700)

ed_plot.plot_interactive_regression_with_color(    
        clean,
        var1, var2,
        'time_of_day', plot_size=700)

In [ ]:
# does albedo have data when other variables are na
mask = ((clean.SW_IN_1_1_1.isna()) | (clean.SW_OUT_1_1_1.isna())) & (~clean.ALB_1_1_1.isna())
subset1 = clean.loc[mask, ['SW_IN_1_1_1', 'SW_OUT_1_1_1', 'ALB_1_1_1']]
print(f'ALB 1: {len(subset1)} records have ALB but missing one or more component values, equal to {(len(subset1)/len(mask)*100)} of records')

mask = ((clean.SW_IN_1_1_2.isna()) | (clean.SW_OUT_1_1_2.isna())) & (~clean.ALB_1_1_2.isna())
subset1 = clean.loc[mask, ['SW_IN_1_1_2', 'SW_OUT_1_1_2', 'ALB_1_1_2']]
print(f'ALB 2: {len(subset1)} records have ALB but missing one or more component values, equal to {(len(subset1)/len(mask)*100)} of records')

## Final Plots

In [ ]:
vars = ['NETRAD',"LW_IN", 'LW_OUT', "SW_IN", 'SW_OUT', 'ALB']
subset_type = 'clean'
color_col = 'days_since_20240101'

for var in vars:
    var1 = f'{var}_1_1_1'
    var2 = f'{var}_1_1_2'
    output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
    color_col = 'days_since_20240101'

    ed_plot.plot_linear_regression_with_color(clean, var1, var2, color_col, output_name, print_plot=True)


In [ ]:
var1 = 'ALB_1_1_1'
var2 = 'ALB_1_1_2'
subset_type = 'clean'
color_col = 'SW_IN_1_1_1'
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}_by_SW_IN.png'
ed_plot.plot_linear_regression_with_color(clean, var1, var2, color_col, output_name, print_plot=True)


In [ ]:
var1 = f'PPFD_IN_1_1_1'
var2 = f'SW_IN_1_1_1'
color_col = 'days_since_20240101'
subset_type = 'clean'

output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
ed_plot.plot_linear_regression_with_color(clean, var1, var2, color_col, output_name, print_plot=True)

var2 = f'SW_IN_1_1_2'
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
ed_plot.plot_linear_regression_with_color(clean, var1, var2, color_col, output_name, print_plot=True)

In [ ]:
# daily comparison
# weird spkes at start of records are b/c data are incomplete
daily_all = clean[['NETRAD_1_1_1', 'NETRAD_1_1_2']].resample('D').mean()
daily_all = daily_all.replace(0,np.nan)

ed_plot.plotlystuff([daily_all, daily_all], ['NETRAD_1_1_1', 'NETRAD_1_1_2'])

# Wind Speed and Direction

## Wind Speed

In [ ]:
# view any lag
ws_offset = validate.detect_sectional_offsets_indexed(clean, clean,
                                          'WS_1_1_1', 'WS_1_1_2')
validate.plot_sectional_lags_plotly(ws_offset)

In [ ]:
# review timeseries for each variable
ed_plot.plotlystuff([clean, clean], ['WS_1_1_1', 'WS_1_1_2'], chrttitle='Wind Speed')
ed_plot.plot_interactive_regression_with_color(
    clean, 'WS_1_1_1', 'WS_1_1_2', 'days_since_20240101', plot_size=700)

In [ ]:
vars = ['WS']
subset_type = 'clean'
color_col = 'time_of_day'

for var in vars:
    var1 = f'{var}_1_1_1'
    var2 = f'{var}_1_1_2'
    output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
    ed_plot.plot_linear_regression_with_color(clean, var1, var2, color_col, output_name, print_plot=True)

## Wind Direction Plots

In [ ]:
from windrose import WindroseAxes

In [ ]:
# # fix old azimuth by adding 10 degres
# # was recorded as 217, but measured as 227 (and 233...)
# old_azimuth = 217
# new_azimuth = 227
# diff = new_azimuth - old_azimuth
# clean['WD_1_1_1'] = (clean.WD_1_1_1+diff)% 360

In [ ]:
# review timeseries for each variable
ed_plot.plotlystuff([clean, clean], ['WD_1_1_1', 'WD_1_1_2'], 
                    chrttitle='Wind Direction')
ed_plot.plot_interactive_regression_with_color(
    clean, 'WD_1_1_1', 'WD_1_1_2', 'time_of_day', plot_size=700)

In [ ]:
clean['wd_diff_correct'] = ((clean['WD_1_1_1'] -clean['WD_1_1_2'] + 180) % 360) - 180
wind_diff = clean.wd_diff_correct.median()
print(wind_diff)
wind_diff_final = -80

# can play around with the filtering, just looking for changes in wind diff over time
mask = (clean.wd_diff_correct>-100) & (clean.wd_diff_correct<-60)
temp = clean[mask]
ed_plot.plotlystuff([clean], ['wd_diff_correct'])

#clean['WD_1_1_2'] = (clean.WD_1_1_2+wind_diff_final)% 360

In [ ]:
# wind rose plots

# drop dates where data missing from either instrument 
vars = ['WD_1_1_2', 'WD_1_1_1', "WS_1_1_1", 'WS_1_1_2']
wind_df = clean[vars].dropna(axis=0, how='any')
wind_df.resample('ME').size()

wind_df['YearMonth'] = wind_df.index.to_period('M')
grouped_data = wind_df.groupby('YearMonth')

for name, group_df in grouped_data:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), subplot_kw={'projection': 'windrose'})

    ax1.set_title(f'Young anemometer for {name}')
    ax1.bar(group_df['WD_1_1_2'], group_df['WS_1_1_2'], normed=True, opening=0.8, edgecolor='white')
    ax1.set_legend()

    ax2.set_title(f'Irgason anemometer for {name}')
    ax2.bar(group_df['WD_1_1_1'], group_df['WS_1_1_1'], normed=True, opening=0.8, edgecolor='white')
    ax2.set_legend()

    plt.tight_layout()
    #plt.savefig(f'{station_name}_plots/{station}_WINDROSE_{name}.png')

# Soil Heat Flux

## Plotly stuff plots

In [ ]:
clean.SWC_3_2_1.describe()

In [ ]:
ed_plot.plotlystuff([clean, clean, clean, clean], ['SWC_1_1_1', 'SWC_2_1_1', 'SWC_3_1_1', 'SWC_3_2_1'], 
                    chrttitle='SWC Dugout')

In [ ]:
# review timeseries for each variable
ed_plot.plotlystuff([clean, clean, clean], ['TS_1_1_1', 'TS_2_1_1', 'TS_3_1_1'], 
                    chrttitle='TS')
ed_plot.plotlystuff([clean, clean, clean], ['SWC_1_1_1', 'SWC_2_1_1', 'SWC_3_1_1'], 
                    chrttitle='SWC')
ed_plot.plotlystuff([clean, clean, clean], ['G_PLATE_1_1_1', 'G_PLATE_2_1_1','G_SOILVUE'], 
                    chrttitle='G PLATE')
ed_plot.plotlystuff([clean, clean, clean], ['SG_1_1_1', 'SG_2_1_1','SG_3_1_1'], 
                    chrttitle='SG')
ed_plot.plotlystuff([clean, clean, clean], ['G_1_1_1', 'G_2_1_1', 'G_3_1_1'], 
                    chrttitle='G')

In [ ]:
plotlystuff_test([clean, clean],
                 ['SWC_1_1_1','G_1_1_1'],
                 two_yaxes=True)

## Linear Regression Plots

In [ ]:
vars = ['TS', 'SWC', 'G_PLATE', 'SG']
color_col = 'days_since_20240101'

for var in vars:
    var1 = f'{var}_1_1_1'
    var2 = f'{var}_2_1_1'

    ed_plot.plot_interactive_regression_with_color(    
        clean,
        var1, var2,
        color_col, 
        plot_size=700)

In [ ]:
# compare all G values
color_col = 'days_since_20240101'
ed_plot.plot_interactive_regression_with_color(    
        clean,
        'G_1_1_1', 'G_2_1_1',
        color_col, plot_size=700)

ed_plot.plot_interactive_regression_with_color(    
        clean,
        'G_1_1_1', 'G_3_1_1',
        color_col, plot_size=700)

ed_plot.plot_interactive_regression_with_color(    
        clean,
        'G_2_1_1', 'G_3_1_1',
        color_col, plot_size=700)

In [ ]:
change_date = pd.to_datetime('2025-07-11 00:00')
earlydf = clean[clean.index<change_date]
latedf = clean[clean.index>=change_date]

color_col = 'days_since_20240101'
var1 = 'SG_1_1_1'
var2 = 'SG_2_1_1'


ed_plot.plot_interactive_regression_with_color(    
        earlydf,
        var1, var2,
        color_col, plot_size=700)

ed_plot.plot_interactive_regression_with_color(    
        latedf,
        var1, var2,
        color_col, plot_size=700)

## Daily Values

In [ ]:
daily_dat = clean[['G_1_1_1', 'G_2_1_1', 'G_3_1_1', 'days_since_20240101']].resample('D').mean()
daily_dat['day_of_year'] = daily_dat.index.dayofyear
subset_type = 'daily'
color_col = 'days_since_20240101'

var1 = 'G_1_1_1'
var2 = 'G_2_1_1'
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
ed_plot.plot_linear_regression_with_color(daily_dat, var1, var2, color_col, 
                                            output_name, print_plot=False)

var1 = 'G_1_1_1'
var2 = 'G_3_1_1'
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
ed_plot.plot_linear_regression_with_color(daily_dat, var1, var2, color_col, 
                                            output_name, print_plot=False)

var1 = 'G_2_1_1'
var2 = 'G_3_1_1'
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
ed_plot.plot_linear_regression_with_color(daily_dat, var1, var2, color_col, 
                                            output_name, print_plot=False)

## Final Plots

In [ ]:
vars = ['TS', 'SWC', 'G_PLATE', 'SG', 'G']
subset_type = 'clean'
color_col = 'days_since_20240101'

for var in vars:
    var1 = f'{var}_1_1_1'
    var2 = f'{var}_2_1_1'
    output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
    
    ed_plot.plot_linear_regression_with_color(clean, var1, var2, color_col, 
                                              output_name, print_plot=True)
    
var1 = 'G_1_1_1'
var2 = 'G_3_1_1'
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
ed_plot.plot_linear_regression_with_color(clean, var1, var2, color_col, 
                                            output_name, print_plot=True)

var1 = 'G_2_1_1'
var2 = 'G_3_1_1'
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
ed_plot.plot_linear_regression_with_color(clean, var1, var2, color_col, 
                                            output_name, print_plot=True)

# Temperature

- TA_1_1_1 EC100 107 Temperature probe (connected to IRGASON)
- TA_1_1_2 Uses Sonic Temperature
- TA_1_1_3 Uses Aspirated Temp/RH probe (EE08) 
- TA_1_1_4 Lil Yellow thing in the Aspirated shield 

In [ ]:
ed_plot.plotlystuff([clean, clean, clean, clean, clean], 
                    ['T_SONIC_1_1_1','TA_1_1_1', 'TA_1_2_1', 'TA_1_3_1','TA_1_4_1'], 
                    chrttitle='Air Temperature')

In [ ]:
# review linear regression plots and then look at residual plots for any that 
# are particularly interesting
vars = ['T_SONIC_1_1_1', 'TA_1_2_1','TA_1_4_1']
color_col = 'H2O_SIG_STRGTH_MIN_1_1_1'

var1 = 'TA_1_3_1'
for var in vars:
    var2 = var
    ed_plot.plot_interactive_regression_with_color(    
        clean,
        var1, var2,
        color_col, plot_size=700)

In [ ]:
vars = ['T_SONIC_1_1_1', 'TA_1_1_1', 'TA_1_2_1','TA_1_4_1']
color_col = 'H2O_SIG_STRGTH_MIN_1_1_1'
subset_type = 'clean'

var1 = 'TA_1_3_1'
for var in vars:
    var2 = var
    output_name = f'{station_name}_plots/{station}_{var1}_{var2}_{subset_type}.png'
    ed_plot.plot_linear_regression_with_color(clean, var1, var2, color_col, output_name, print_plot=True)

# RH

## Relative Humidity

- RH_1_1_1 EC100 107 Temperature probe (connected to IRGASON)
- RH_1_1_2 Uses Sonic Temperature
- RH_1_1_3 Uses Aspirated Temp/RH probe (EE08) 


In [ ]:
ed_plot.plotlystuff([clean, clean, clean], 
                    ['RH_1_1_1', 'RH_1_2_1', 'RH_1_3_1'], 
                    chrttitle='Relative Humidity')

In [ ]:
clean['RH_DIFF'] = clean['RH_1_1_1'] - clean['RH_1_3_1']
color_col = 'H2O_SIG_STRGTH_MIN_1_1_1'
clean.plot.scatter(x='RH_DIFF', y='H2O_SIG_STRGTH_MIN_1_1_1', alpha=0.01)
plt.vlines(x=0, ymin=0.3, ymax=1, color='red', linestyle='--')
plt.show()
#plotlystuff_test([clean, clean], ['RH_DIFF', 'H2O_SIG_STRGTH_MIN_1_1_1'],two_yaxes=True)

In [ ]:
color_col = 'H2O_SIG_STRGTH_MIN_1_1_1'

ed_plot.plot_interactive_regression_with_color(    
        clean,
        'RH_1_1_1',
        'RH_1_2_1',
        color_col, plot_size=700)

ed_plot.plot_interactive_regression_with_color(    
        clean,
        'RH_1_3_1',
        'RH_1_1_1',
        color_col, plot_size=700)

ed_plot.plot_interactive_regression_with_color(    
        clean,
        'RH_1_3_1',
        'RH_1_2_1',
        color_col, plot_size=700)

In [ ]:
# review RH by signal strength
sig = 0.8
low_sig = clean[clean.H2O_SIG_STRGTH_MIN_1_1_1<sig]
high_sig = clean[clean.H2O_SIG_STRGTH_MIN_1_1_1>=sig]

print(f'{low_sig.RH_1_1_1.notna().sum()} records with low signal strength ({sig})')
print(f'{high_sig.RH_1_1_1.notna().sum()} records with high signal strength ({sig})')

var1 = 'RH_1_3_1'
var2 = 'RH_1_1_1'

# data with low signal only
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_lowsig{sig}.png'
ed_plot.plot_linear_regression_with_color(low_sig, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=False)

# high site data
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_highsig{sig}.png'
ed_plot.plot_linear_regression_with_color(high_sig, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=False)



In [ ]:
# Review by flags I creaeted previously
# review RH by signal strength
flagged_bad = clean[clean.H2O_1_1_1_FLAG>=2]
flagged_better = clean[clean.H2O_1_1_1_FLAG<2]

print(f'{flagged_bad.RH_1_1_1.notna().sum()} records with bad flags')
print(f'{flagged_better.RH_1_1_1.notna().sum()} records with good flags')

var1 = 'RH_1_3_1'
var2 = 'RH_1_1_1'

# data with low signal only
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_flagged.png'
ed_plot.plot_linear_regression_with_color(flagged_bad, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=False)

# high site data
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_notflagged.png'
ed_plot.plot_linear_regression_with_color(flagged_better, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=False)

## VPD

In [ ]:
var1 = 'RH_1_3_1'
var2 = 'VPD_1_1_1'
output_name = f'{station_name}_plots/{station}_{var1}_{var2}.png'
ed_plot.plot_linear_regression_with_color(clean, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=True)

var1 = 'RH_1_1_1'
output_name = f'{station_name}_plots/{station}_{var1}_{var2}.png'
ed_plot.plot_linear_regression_with_color(clean, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=True)

In [ ]:
ed_plot.plotlystuff([clean], ['VPD_1_1_1'])

# LE and ET

In [ ]:
# closure plots
clean['NR1_G3'] = (clean['NETRAD_1_1_1'] - clean['G_3_1_1'])
clean['H_plus_LE'] = (clean['H_1_1_1'] + clean['LE_1_1_1'])

In [ ]:
plotlystuff_test([clean, clean], ['H2O_1_1_1_FLAG', 'H2O_SIG_STRGTH_MIN_1_1_1'], two_yaxes=True)
temp = clean[clean.H2O_1_1_1_FLAG==2]
plotlystuff_test([clean, temp], ['H2O_1_1_1', 'H2O_1_1_1'])
plotlystuff_test([clean, temp], ['LE_1_1_1', 'LE_1_1_1'])
plotlystuff_test([clean, temp], ['VPD_1_1_1', 'VPD_1_1_1'])

In [ ]:
sig = 0.8

low_sig = clean[clean.H2O_SIG_STRGTH_MIN_1_1_1<sig]
high_sig = clean[clean.H2O_SIG_STRGTH_MIN_1_1_1>=sig]

flagged_bad = clean[clean.H2O_1_1_1_FLAG>=2]
flagged_better = clean[clean.H2O_1_1_1_FLAG<2]

var1 = 'NR1_G3'
var2 = 'H_plus_LE'


print('All Data')
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_all.png'
ed_plot.plot_linear_regression_with_color(clean, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=True)

print('Straight Threshold Low Signal Data')
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_lowsig{sig}.png'
ed_plot.plot_linear_regression_with_color(low_sig, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=True)

print('Flagged 2')
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_lowsigrolling{sig}.png'
ed_plot.plot_linear_regression_with_color(flagged_bad, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=True)

print('Straight Threshold High Signal Data')
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_highsig{sig}.png'
ed_plot.plot_linear_regression_with_color(high_sig, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=True)

print('Flag 0 or 1')
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_highsigrolling{sig}.png'
ed_plot.plot_linear_regression_with_color(flagged_better, var1, var2, 
                                          'H2O_SIG_STRGTH_MIN_1_1_1', output_name, 
                                          print_plot=True)

In [ ]:
# daily plot of everything
#df = rolling_high_sig.copy()
df = clean.copy()
record_count = 44
color_col = 'H2O_SIG_STRGTH_MIN_1_1_1'
var1 = 'NR1_G3'
var2 = 'H_plus_LE'


print ('Daily All Data')
daily_dat = df[['NR1_G3', 'H_plus_LE', 'H2O_SIG_STRGTH_MIN_1_1_1', 'H2O_1_1_1_FLAG']].resample('D').mean()
daily_count = df.resample('D').count()
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_daily_all.png'
ed_plot.plot_linear_regression_with_color(daily_dat, var1, var2, 
                                          color_col, "", 
                                          print_plot=False)
print('Daily Full Records Only')
count_mask = (daily_count['NR1_G3']>=record_count) & (daily_count['H_plus_LE']>=record_count)
print(f'{(count_mask.sum()/len(count_mask)).round(2)*100} percent of {len(count_mask)} days with complete data for both NR1_G3 and H_plus_LE')
daily_dat_complete = daily_dat[count_mask]
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_daily_completedays.png'
ed_plot.plot_linear_regression_with_color(daily_dat_complete, var1, var2, 
                                          color_col, "", 
                                          print_plot=False)

print('Daily Full Records Only, Not Flagged')
flag_mask = daily_dat_complete['H2O_1_1_1_FLAG']<1.01
not_flagged_daily = daily_dat_complete[flag_mask]
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_daily_not_flagged.png'
ed_plot.plot_linear_regression_with_color(not_flagged_daily, var1, var2, 
                                          color_col, "", 
                                          print_plot=False)


print('Daily Sig Strenght Above Threshold for Complete Days')
flag_mask = daily_dat_complete['H2O_SIG_STRGTH_MIN_1_1_1']>=0.8
daily_strong_sig = daily_dat_complete[flag_mask]
output_name = f'{station_name}_plots/{station}_{var1}_{var2}_daily_strong_sig.png'
ed_plot.plot_linear_regression_with_color(daily_strong_sig, var1, var2, 
                                          color_col, "", 
                                          print_plot=False)

In [ ]:
# Plot where daily sig strenght is above a threshold
df = clean.copy()
record_count = 44
sig = 0.8
color_col = 'H2O_SIG_STRGTH_MIN_1_1_1'
var1 = 'NR1_G3'
var2 = 'H_plus_LE'


daily_dat = df[['NR1_G3', 'H_plus_LE', 'H2O_SIG_STRGTH_MIN_1_1_1', 'H2O_1_1_1_FLAG']].resample('D').mean()
daily_count = df.resample('D').count()
count_mask = (daily_count['NR1_G3']>=record_count) & (daily_count['H_plus_LE']>=record_count)
daily_dat_complete = daily_dat[count_mask]
print(f'Percent of records that pass the number of records threshold: {len(daily_dat_complete)/len(daily_dat)*100:.1f}%',  f'{len(daily_dat_complete)} records')

flag_mask = daily_dat_complete['H2O_SIG_STRGTH_MIN_1_1_1']>=sig
daily_strong_sig = daily_dat_complete[flag_mask]

print(f'Percent of complete records that both signal threshold: {len(daily_strong_sig)/len(daily_dat_complete)*100:.1f}%',  f'{len(daily_strong_sig)} records')

output_name = f'{station_name}_plots/{station}_{var1}_{var2}_daily_strong_sig.png'
ed_plot.plot_linear_regression_with_color(daily_strong_sig, var1, var2, 
                                          color_col, "", 
                                          print_plot=False)

In [ ]:

# 1. Ensure index is datetime
df.index = pd.to_datetime(df.index)

# 2. Filter for non-NA records in LE_1_1_1
filtered_df = df[df['LE_1_1_1'].notna()]

# 3. Get the counts of each flag per day
counts = (filtered_df.groupby([filtered_df.index.date, 'H2O_1_1_1_FLAG'])
          .size()
          .unstack(fill_value=0))

# 4. Get the daily mean of signal strength
# Replace 'sig_strength' with your actual column name
daily_mean_sig = filtered_df.groupby(filtered_df.index.date)['H2O_SIG_STRGTH_MIN_1_1_1'].mean()

# 5. Combine them into one dataframe
result = counts.assign(mean_signal_strength=daily_mean_sig)

# Clean up the index name
result.index.name = 'Date'


#result.to_csv('daily_le_values.csv')

# CO2

In [ ]:
plotlystuff_test([clean, clean, clean],
                    ['CO2_1_1_1','CO2_SIG_STRGTH_MIN_1_1_1',  'FC_1_1_1'], 
                    chrttypes=['lines', 'lines', 'lines'], 
                    two_yaxes=True)

In [ ]:
plotlystuff_test([clean, clean], ['CO2_1_1_1_FLAG', 'CO2_SIG_STRGTH_MIN_1_1_1'], two_yaxes=True)
temp = clean[clean.CO2_1_1_1_FLAG==2]
plotlystuff_test([clean, temp], ['CO2_1_1_1', 'CO2_1_1_1'])